<a href="https://colab.research.google.com/github/robertbarcik/ADK-tutorial/blob/main/notebooks/03_sessions_state.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 03 — Sessions, State, Events, Artifacts

> **Where you are** — you know agents + tools (M01–M02); your old `messages` list became `Session`.
> - **You can already:** keep conversation history as a list of message dicts — that's what a session stores for you.
> - **New in this module:** the state dict that rides along with the history, prefixes that decide what survives, and events as the record of everything.
> - **No new Python** — one new ADK idea: ADK *injecting* an argument into your tool (explained when we get there).

If Module 02 was about what an agent can *do*, Module 03 is about what an agent *remembers*.

Start with the everyday problem. A user talks to your agent today and comes back tomorrow. For the agent to feel like a colleague and not a goldfish, something has to be **stored** in the meantime — and that one word immediately raises very practical questions:

- **Where?** In RAM? In a file? In a database?
- **What exactly?** The whole conversation word for word — or just the useful facts pulled out of it, like *favorite color: teal*?
- **For whom?** Should tomorrow's conversation see it? Should *other users'* conversations?

Here is the good news: for the conversation itself you already have the answer — the **Session** you've been using since M01 stores the whole dialogue. This module opens that box, adds the second kind of memory (the extracted facts, kept in **state**), and shows the five-character trick that answers "for whom, and for how long".

**What we'll do:**

1. Open the Session box: what exactly it holds.
2. Meet **state prefixes** — the naming convention that decides how long each piece of data lives.
3. The wow demo: the same user opens a **second, separate session** — and the agent still remembers them.
4. The three ways to write state — and the one natural-looking way that silently *doesn't* persist.
5. Walk the event history, and a short word on artifacts (big files).

**Running cost:** under $0.01 on OpenRouter.

# Setup

Same ritual as every module: install, key, imports.

In [1]:
!pip install -q google-adk==2.7.1 litellm==1.85.7 python-dotenv==1.0.1 nest-asyncio==1.6.0 deprecated==1.2.18 2>/dev/null

print("✅ Packages installed.")

✅ Packages installed.


Same OpenRouter key — picked up from Colab secrets or `.env`.

In [2]:
import os

OPENROUTER_API_KEY = None
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
    print("✅ API key loaded from Colab secrets.")
except Exception:
    try:
        from dotenv import load_dotenv
        load_dotenv()
        OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
        if OPENROUTER_API_KEY:
            print("✅ API key loaded from .env file.")
    except ImportError:
        pass

if not OPENROUTER_API_KEY:
    from getpass import getpass
    print("💡 Set OPENROUTER_API_KEY in Colab secrets (🔑 icon) or a local .env file.")
    OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")

assert OPENROUTER_API_KEY, "❌ No API key provided."
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

MODEL_STRING = "openrouter/openai/gpt-5.6-luna"
print(f"✅ Model: {MODEL_STRING}")

✅ API key loaded from .env file.
✅ Model: openrouter/openai/gpt-5.6-luna


One import is new: `ToolContext` — you'll meet it in the first demo.

In [3]:
import os
import sys, warnings, asyncio, uuid, logging
warnings.filterwarnings("ignore")
try:
    sys.stderr.fileno()
except Exception:
    sys.stderr = open(os.devnull, "w")

import nest_asyncio; nest_asyncio.apply()
os.environ.setdefault("LITELLM_LOG", "ERROR")  # silence LiteLLM's import-time provider warnings
import litellm; litellm.suppress_debug_info = True
logging.getLogger("LiteLLM").setLevel(logging.WARNING)

from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.models.lite_llm import LiteLlm
from google.adk.tools.tool_context import ToolContext
from google.genai import types

print("✅ Imports successful.")

✅ Imports successful.


# What a Session Holds

The `Session` from M01, one level deeper. ADK files every conversation under a triple: `(app_name, user_id, session_id)` — which app, which user, which of their conversations. Inside, a session holds two things:

- **A list of events** — the full ordered history: every message, tool call, tool response. ADK appends to it; your code can read it.
- **A state dict** — an ordinary key-value store for the *extracted facts*: `{"favorite_color": "teal"}`. This dict is the module's main character.

And where does all of this physically live? So far: **in RAM.** `InMemorySessionService` is literally a Python dict inside your notebook process — close the notebook and everything is gone. That is deliberately naive, and perfectly fine for learning. In M08 we swap it for a real database *with one changed line*, and nothing else you learn today changes — so storage is not our worry yet.

# State Prefixes — Who Remembers What, and For How Long

Say your agent learns the user's favorite color. Should it still know it tomorrow, in a new conversation? Should *other* users' agents know it? Obviously not the same answer for every piece of data — and ADK's answer is a naming convention that is easy to miss in the docs and useful every single day:

**The key's prefix decides where the value lives and how long it survives.**

| Prefix | Scope | Survives |
|---|---|---|
| *(none)* | This session only | Until the session is deleted |
| `user:` | This user, across all their sessions | As long as the user exists |
| `app:` | Global across all users of this app | As long as the app exists |
| `temp:` | This invocation only | Thrown away after the current run |

Four rings, from throwaway scratch to global settings — chosen with five characters at the front of a dict key. The demo makes it concrete.

# Writing State From a Tool

We want the agent to **remember the user's favorite color**. How does the color get into the state dict? With a tool — an ordinary tool, exactly like the ones you built in M02:

```python
def remember_favorite_color(color: str, tool_context: ToolContext) -> dict:
    tool_context.state["user:favorite_color"] = color
```

The model does what it always does with tools: it hears *"my favorite color is teal"*, decides this tool fits, extracts `color='teal'`, and calls it. Inside, the function writes the value into state — a plain dict assignment, with the `user:` prefix so it survives into future sessions. And recall is the same trick in the other direction: a second tool reads the dict and returns the value to the model.

### Two details before you run it

**Who fills in `tool_context`? ADK — not the model.** ADK sees the `ToolContext` type hint in the signature and slips the object in at call time; the model never even sees that parameter in the schema. It is your ticket from inside a tool to the running session. (It's the mirror image of the `**arguments` unpacking you did by hand in the previous course: there you routed the model's arguments *into* the function — here ADK adds one extra, non-model argument alongside them.)

**The agent below also writes state, on its own.** It is defined with `output_key="last_response"`, which tells ADK: after every reply, save the agent's final text into state under this key. So watch for **two** writes in the demo — the tool's (prefixed) and `output_key`'s (unprefixed).

In [4]:
APP = "m03_demo"
USER = "alice"
session_service = InMemorySessionService()

def remember_favorite_color(color: str, tool_context: ToolContext) -> dict:
    """Record the user's favorite color for future sessions.

    Args:
        color: A color name like "teal", "crimson", "forest green".
    """
    # Note the user: prefix — this makes the value survive beyond this session.
    tool_context.state["user:favorite_color"] = color
    return {"status": "saved", "color": color}


def recall_favorite_color(tool_context: ToolContext) -> dict:
    """Check whether the user has told us their favorite color before."""
    color = tool_context.state.get("user:favorite_color")
    if color:
        return {"known": True, "color": color}
    return {"known": False}


color_agent = LlmAgent(
    name="color_agent",
    model=LiteLlm(model=MODEL_STRING),
    description="Remembers and recalls the user's favorite color across sessions.",
    instruction=(
        "You track the user's favorite color. "
        "When they tell you a color, call remember_favorite_color. "
        "When they ask what their color is, call recall_favorite_color first. "
        "Be brief."
    ),
    tools=[remember_favorite_color, recall_favorite_color],
    output_key="last_response",   # also save the final text to state['last_response']
)

print("✅ color_agent ready.")

✅ color_agent ready.


## Run It — Session 1

Our `chat()` helper gains one new argument: `sid`, the session id. Until now every call created a fresh session; from here we **reuse** sessions — that's the whole point of this module. The first lines of the helper do get-or-create: fetch the session, create it only if it doesn't exist yet.

At the end of the cell we fetch the session back and print its state — our first look inside.

In [5]:
async def chat(agent, prompt: str, sid: str):
    sess = await session_service.get_session(app_name=APP, user_id=USER, session_id=sid)
    if sess is None:
        await session_service.create_session(app_name=APP, user_id=USER, session_id=sid)
    runner = Runner(agent=agent, app_name=APP, session_service=session_service)
    message = types.Content(role="user", parts=[types.Part(text=prompt)])
    print(f"USER ({sid}): {prompt}")
    async for event in runner.run_async(user_id=USER, session_id=sid, new_message=message):
        if event.content and event.content.parts:
            for p in event.content.parts:
                if p.text and p.text.strip():
                    tag = "[FINAL]" if event.is_final_response() else "[step]"
                    print(f"{tag} {event.author}: {p.text.strip()[:200]}")
                if p.function_call:
                    print(f"[tool_call] {p.function_call.name}({dict(p.function_call.args)})")
                if p.function_response:
                    print(f"[tool_resp] {p.function_response.response}")
    print()

SID_1 = "session-one"
await chat(color_agent, "My favorite color is teal.", SID_1)

# Inspect what's in the session's state now.
s1 = await session_service.get_session(app_name=APP, user_id=USER, session_id=SID_1)
print("── Session 1 state after the conversation ──")
for k, v in sorted(dict(s1.state).items()):
    print(f"  {k!r}: {v!r}")

USER (session-one): My favorite color is teal.


[tool_call] remember_favorite_color({'color': 'teal'})
[tool_resp] {'status': 'saved', 'color': 'teal'}


[FINAL] color_agent: Got it.

── Session 1 state after the conversation ──
  'last_response': 'Got it.'
  'user:favorite_color': 'teal'


### 🔍 What just happened?

Two separate writes landed in state:

- The tool wrote `user:favorite_color = 'teal'` — you can see the `[tool_call]` in the stream.
- `output_key="last_response"` auto-saved the model's final sentence under `last_response`.

Both are now part of the session. Now the real test: a **completely separate session** for the same user. The `user:`-prefixed key should carry over. The unprefixed `last_response` should not.

## Session 2 — Same User, Fresh Session

This is the moment the module exists for. We create session two with no initial state at all, print what it starts with, and then ask the agent a question it can only answer if the prefix system works.

In [6]:
SID_2 = "session-two"

# Create the second session fresh — notice we don't pass any initial state.
await session_service.create_session(app_name=APP, user_id=USER, session_id=SID_2)
s2_initial = await session_service.get_session(app_name=APP, user_id=USER, session_id=SID_2)

print("── Session 2 initial state (before any chat) ──")
for k, v in sorted(dict(s2_initial.state).items()):
    print(f"  {k!r}: {v!r}")
print()

# Ask the agent in session 2 — does it remember?
await chat(color_agent, "Hey, what's my favorite color?", SID_2)

── Session 2 initial state (before any chat) ──
  'user:favorite_color': 'teal'

USER (session-two): Hey, what's my favorite color?


[tool_call] recall_favorite_color({})
[tool_resp] {'known': True, 'color': 'teal'}


[FINAL] color_agent: Your favorite color is teal.



### 🔍 What just happened?

Read the output top to bottom:

- Session 2 **started** with `user:favorite_color: 'teal'` already present — it survived.
- `last_response` from session 1 is **not** there — unprefixed, so it stayed behind.
- The agent called `recall_favorite_color`, read the surviving key, and answered "teal".

That is cross-session memory in 80 lines of code. No database, no vector store — a prefix convention on a dict key. You decide the lifetime of every piece of data with five characters, and ADK handles the rest.

### 🎯 Mini-tasks

1. **App scope.** Write a tool `set_app_mode(mode: str, tool_context: ToolContext)` that stores `tool_context.state["app:mode"] = mode`. Create two different users (change `USER` between calls). Does `app:mode` show up for both?
2. **Temp scope.** Write a tool that stores `tool_context.state["temp:scratch"] = "..."`. After the run completes, fetch the session and inspect the state. Is `temp:scratch` still there?

# Three Ways to Write State — Only Two Persist

So far state has been written in two ways, and both stuck:

1. **From inside a tool:** `tool_context.state["user:favorite_color"] = "teal"`.
2. **Automatically, by the agent:** `output_key="last_response"` saved the final reply.

There is a third way that *looks* the most natural of all: fetch the session and assign into its state directly — `session.state["user:mood"] = "happy"`. **It silently does not persist.** No error, no warning; the next fetch simply returns the old state.

Why? ADK stores state **through events**: each of the two working patterns records a small event ("this key changed to this value"), and the state dict you see is rebuilt from those records. Direct assignment writes into a copy in your hands and records nothing — so nothing survives.

The rule: **write state from a tool, or via `output_key`. Never assign into a fetched session and expect it to stick.** The next cell walks into the trap on purpose, so you recognise it later in your own code.

In [7]:
# Demonstrate the pitfall — do NOT copy this pattern.
# Mutate state directly and re-fetch to prove it didn't stick.
sess = await session_service.get_session(app_name=APP, user_id=USER, session_id=SID_2)
sess.state["user:mood"] = "happy"                 # direct assignment — WRONG
sess.state["this_does_not_persist"] = "ghost"     # unprefixed — also wrong, but doubly so

sess_fresh = await session_service.get_session(app_name=APP, user_id=USER, session_id=SID_2)
print("── Re-fetched session state ──")
for k, v in sorted(dict(sess_fresh.state).items()):
    print(f"  {k!r}: {v!r}")
print()
print("Notice: 'user:mood' and 'this_does_not_persist' are NOT in the re-fetched state.")
print("Direct assignment to a returned session's .state does not round-trip.")

── Re-fetched session state ──
  'last_response': 'Your favorite color is teal.'
  'user:favorite_color': 'teal'

Notice: 'user:mood' and 'this_does_not_persist' are NOT in the re-fetched state.
Direct assignment to a returned session's .state does not round-trip.


### 🔍 What just happened?

Both direct assignments vanished on re-fetch — `user:mood` *and* the unprefixed key. The prefix didn't matter: no event was recorded, so nothing persisted. When state "mysteriously doesn't save" in your own agent, this is the first thing to check.

# Events — Everything That Happened, In Order

The session's other half is the **event list** — and unlike state, you never write it yourself; ADK has been quietly appending to it all along. Why would you go back and *read* it? Three everyday reasons:

- A user says *"the agent told me something wrong yesterday"* — you replay the conversation and see exactly which tool returned what.
- You want a clean transcript of a conversation — you rebuild it from the events.
- You want to test the agent — the events are the record you check against (M09 does exactly this).

The one-line version: **a session's event list is the agent's log file, with structure.** The next cell walks session 1's history and labels each event.

In [8]:
# Show the event history of session 1.
s1 = await session_service.get_session(app_name=APP, user_id=USER, session_id=SID_1)
print(f"Session 1 has {len(s1.events)} events.\n")
for i, ev in enumerate(s1.events):
    author = ev.author or "(system)"
    kinds = []
    if ev.content and ev.content.parts:
        for p in ev.content.parts:
            if p.text and p.text.strip():
                kinds.append(f"text({p.text.strip()[:40]}...)" if len(p.text.strip()) > 40 else f"text({p.text.strip()})")
            if p.function_call:
                kinds.append(f"tool_call({p.function_call.name})")
            if p.function_response:
                kinds.append(f"tool_resp({p.function_response.name if hasattr(p.function_response, 'name') else '?'})")
    if ev.actions and ev.actions.state_delta:
        kinds.append(f"state_delta({list(ev.actions.state_delta.keys())})")
    print(f"  [{i:>2}] {author:16s} {' | '.join(kinds) if kinds else '(no content)'}")

Session 1 has 4 events.

  [ 0] user             text(My favorite color is teal.)
  [ 1] color_agent      tool_call(remember_favorite_color)
  [ 2] color_agent      tool_resp(remember_favorite_color) | state_delta(['user:favorite_color'])
  [ 3] color_agent      text(Got it.) | state_delta(['last_response'])


### 🔍 What just happened?

Notice the `state_delta` entries. Every state write — the tool's, `output_key`'s, session setup — was recorded as an event. The state dict you inspected earlier is not stored separately: **ADK rebuilds it by replaying these deltas in order** whenever you fetch the session.

This design has a name — [event sourcing](https://martinfowler.com/eaaDev/EventSourcing.html): the history is the truth, and the current state is computed from it. Banks and databases work this way; ADK does the same for conversations. It is also exactly why the direct-assignment trap fails: no event, no persistence.

### 🎯 Mini-task

Write a loop that walks `s1.events` and prints only the user's and the agent's *text* messages, in order — a clean, readable transcript with none of the tool machinery.

# Artifacts — Big Files Next to the Conversation

One more concept, briefly. **Artifacts** are for binary data — images, audio, PDFs — that belong to a session but shouldn't be stuffed into the event list. Think of an email attachment: the message text stays small, the file travels alongside it. In ADK, the event stream keeps a small pointer; the payload lives in a separate store.

The services mirror the session services: `InMemoryArtifactService` for demos, `GcsArtifactService` (Google Cloud Storage) for production, or your own implementation for S3 and friends.

For the text-only agents in Part 1 you can ignore artifacts entirely — they matter once an agent accepts an uploaded image or generates a PDF report. It's enough to know the drawer exists.

# Stored Memory Goes Stale — A Warning

> *From the "Agentic Design Patterns" publication, Chapter 2. Two minutes of theory you'll use for the rest of the course.*

The wow demo looked clean because the state had no time to age. In production it ages. A favorite color from three months ago is probably still true. The user's "current project"? Maybe not. The tickets the agent remembered as open yesterday might be closed. **Stored memory is a hint, not a fact.**

Three guidelines from the publication:

1. **Before high-stakes actions, re-check instead of recalling.** If the agent is about to send an email or charge a card based on stored state, have it call a read-only tool to verify first.
2. **Scope aggressively.** `temp:` for scratch, unprefixed for the session, `user:` only for things that change by explicit user action, `app:` only for near-constant configuration.
3. **When writing `user:` or `app:` state, store a short "why" note too.** Six months from now it will save you hours of debugging.

The publication's name for the habit: **Skeptical Memory** — treat your own stored context as unverified until proven otherwise.

### 🎯 Mini-task

Make the agent save a favorite color, then change `session.state["user:favorite_color"]` to something else via a direct (non-persisting) assignment. Ask the agent what the color is. Which value does it see — the persisted one, or your uncommitted change? Explain why using the event-replay picture.

# Key Takeaways

- A **Session** is a triple `(app_name, user_id, session_id)` holding events and a state dict.
- **State prefixes** decide lifetime: `user:` survives across sessions, `app:` is global, `temp:` is per-invocation, unprefixed is per-session.
- **Only two ways to write state persist:** `output_key=` on the agent and `tool_context.state[...]` inside a tool. Direct assignment on a fetched session does NOT stick — no event, no persistence.
- **Events are the record of everything, in order.** State is rebuilt by replaying the state-delta events.
- **Artifacts** hold big files next to the conversation — a pointer in the events, the payload elsewhere.
- **Skeptical Memory:** stored state is a hint, not a fact. Re-check before high-stakes actions.

# Next up — M04: The One-Line Model Swap

You've typed `LiteLlm(model="openrouter/openai/gpt-5.6-luna")` all course. M04 opens that up: the same agent on Claude, GPT, Qwen and a locally-hosted Ollama model — plus the one prefix mistake that causes infinite tool-call loops.